# Israel Doctors EDA — Portfolio review

This notebook reproduces selected **published summary exports** from the repository. It does **not** re-extract PDF tables, independently audit clinical records, run SQL Server, or regenerate the full v22 pipeline. See [Research validation](../RESEARCH_VALIDATION.md) and the [original investigation](Israel_Doctors_EDA_Exploration_22.ipynb).

**Run:** From the `analysis/` directory after installing `requirements.txt`. All charts and tables below read version-controlled CSV exports. Numbers marked PROVISIONAL remain exploratory.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

EXPORTS = Path("exports")
if not EXPORTS.is_dir():
    raise FileNotFoundError("Run this notebook from the analysis/ directory.")

def read_export(filename):
    return pd.read_csv(EXPORTS / filename, encoding="utf-8-sig")

## 1 · Missing-data interpolation sensitivity

The primary 2021 training association is unchanged by interpolation; another pair gains a reporting country and changes modestly. This does not establish correctness of imputed values.

In [ ]:
sensitivity = read_export("OECD_2021_Interpolation_Sensitivity.csv")
display(sensitivity)

## 2 · Physician stock is not the same as practising physicians

The stock-by-training series and the practising-physicians series use different definitions. Their growth rates should not be compared as interchangeable measures.

In [ ]:
growth = read_export("Israel_2010_2024_population_physician_growth_summary.csv")
display(growth)

## 3 · The OECD benchmark depends on country coverage

In 2021, Israel reported 3.39 practising physicians per 1,000 people. The median among 32 reporting OECD countries was 3.79; the fixed 26-country panel median was 3.53. The implied counts are **benchmark-equivalent differences**, not estimates of a clinically justified shortage.

In [ ]:
bench = read_export("OECD_2021_Israel_Practising_Physicians_Benchmarks.csv")
display(bench)
fixed = read_export("OECD_2010_2024_Israel_vs_fixed_medians.csv")
fixed_2021 = fixed[(fixed["year"] == 2021) & (fixed["metric"] == "practising_per_1000")]
display(fixed_2021)

## 4 · Hospital-level correlations (2021)

Only 21–24 hospitals contribute to each correlation. All measures are hospital **process** indicators and remain provisional pending data-definition verification.

In [ ]:
hospital = read_export("MOH_2021_training_quality_correlations_PROVISIONAL.csv")
display(hospital)
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(hospital["metric"], hospital["pearson_r"])
ax.axhline(0, linewidth=1)
ax.set_ylabel("Pearson r")
ax.set_title("Hospital-level associations · exploratory · 2021")
ax.tick_params(axis="x", rotation=15)
plt.tight_layout()
plt.show()

## 5 · Specialty-level correlations (2020): hypotheses, not confirmation

Cardiology/CHADS has a strong nominal correlation in a small sample (n=9). It is not statistically significant after correcting the three reported comparisons at a family-wise 5% level, and specialty extraction, denominator sizes, hospital composition and CHADS responsibility remain unverified.

In [ ]:
specialty = read_export("MOH_2020_specialty_vs_2020_quality_pearson_PROVISIONAL.csv")
display(specialty)
leave_one_out = read_export("MOH_2020_Cardiology_CHADS_Leave_One_Out.csv")
display(leave_one_out.sort_values("pearson_r"))
print("Range of leave-one-hospital-out correlations:", leave_one_out["pearson_r"].min(), leave_one_out["pearson_r"].max())

## 6 · Missing numerators: hip-fracture source-record audit

The audit supports interpreting some blank numerators as zero documented successes; it does not independently confirm every metric or resolve the recorded exactly-48-hour boundary. **Records are not necessarily individual patients.**

In [ ]:
hip_audit = read_export("MOH_Hip_Blank_Numerator_Time_Audit.csv")
display(hip_audit)

## Interpretation checklist

- No ecological hospital-level association proves the quality of an individual physician.
- Training origin is not randomly assigned; differences in hospital resources, patient severity and speciality can confound comparisons.
- Do not confuse the OECD foreign-trained stock with graduates affected by the specific Yatziv reform restrictions.
- A failure to detect statistical significance is **not** evidence that the true relationship must be zero.
- No original SQL scripts are asserted by this review notebook; the legacy database is separate.

For underlying exploration and charts, use the [original notebook](Israel_Doctors_EDA_Exploration_22.ipynb).